In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Read the CSV file
df = pd.read_csv('z_scores_scaled/combined_z_scores.csv', delimiter=',')

# Get list of columns with 'percentile' in the name
percentile_cols = [col for col in df.columns if 'percentile' in col.lower()]

# Create new dataframe with emplid and percentile columns
selected_df = df[['emplid','segment','cluster_name'] + percentile_cols]

# Display first few rows
print(selected_df.head())

# If you want to see all column names
print("\nSelected columns:")
print(selected_df.columns.tolist())


In [3]:
# Transpose the df for processing
skills_df = selected_df.melt(
        id_vars=['emplid','segment','cluster_name'],
        var_name='skill',
        value_name='score'
        )

In [4]:
# Remove '_z_score_percentile' from column names
skills_df['skill'] = skills_df['skill'].str.replace('_z_score_percentile', '')

In [5]:
# File paths
content_file = 'Recommendations/training_content.csv'
consumption_file = 'Recommendations/content_consumption.csv'

In [6]:
# Load the CSV files
content_df = pd.read_csv(content_file, delimiter = '|')
consumption_df = pd.read_csv(consumption_file, delimiter = '|')

In [7]:
# If content_df contains comma-delimited skills
content_df['skill'] = content_df['skill'].str.split(',')
content_df = content_df.explode('skill').reset_index(drop=True)
content_df['skill'] = content_df['skill'].str.strip()

In [8]:
# If consumption_df contains comma-delimited content_ids
consumption_df['content_id'] = consumption_df['content_id'].str.split(',')
consumption_df = consumption_df.explode('content_id').reset_index(drop=True)
consumption_df['content_id'] = consumption_df['content_id'].str.strip()

In [9]:
# Set skills threshold here
# Create a DataFrame with skills scoring less than defined threshold
below_threshold = []
for _, row in skills_df.iterrows():
    if row['score'] < 71: # Enter skills threshold here
        below_threshold.append({
            'emplid': row['emplid'],
            'skill': row['skill'],
            'score': row['score']
        })

In [10]:
below_threshold_df = pd.DataFrame(below_threshold)

# Identify high performers (top 25%) for each skill
percentile = 75
high_performers_df = skills_df[
    skills_df.groupby('skill')['score'].transform(
        lambda x: x >= np.percentile(x, percentile)
    )
]

In [11]:
# Create employee skill matrix
employee_skill_matrix = skills_df.pivot(index='emplid', columns='skill', values='score').fillna(0)

# Create content skill matrix
content_skill_matrix = pd.get_dummies(content_df['skill']).groupby(content_df['content_id']).sum()

# Find common skills
common_skills = set(employee_skill_matrix.columns) & set(content_skill_matrix.columns)

In [12]:
# Filter matrices to include only common skills
employee_skill_matrix = employee_skill_matrix[list(common_skills)]
content_skill_matrix = content_skill_matrix[list(common_skills)]

In [13]:
# Normalize matrices
scaler = MinMaxScaler()
employee_skill_matrix_norm = pd.DataFrame(scaler.fit_transform(employee_skill_matrix), 
                                          columns=employee_skill_matrix.columns, 
                                          index=employee_skill_matrix.index)
content_skill_matrix_norm = pd.DataFrame(scaler.fit_transform(content_skill_matrix), 
                                         columns=content_skill_matrix.columns, 
                                         index=content_skill_matrix.index)

In [14]:
# Print shapes for debugging
print("Employee skill matrix shape:", employee_skill_matrix_norm.shape)
print("Content skill matrix shape:", content_skill_matrix_norm.shape)
print("Number of common skills:", len(common_skills))
print("Common skills:", sorted(list(common_skills)))

Employee skill matrix shape: (7669, 11)
Content skill matrix shape: (171, 11)
Number of common skills: 11
Common skills: ['Account Management', 'Communication', 'Customer Engagement', 'Industry Knowledge', 'Problem Solving', 'Process', 'Project Management', 'Prospecting', 'Sales Skills', 'Strategy', 'Technology']


In [ ]:
# Generate recommendations
recommendations = []

# For each employee with skills below threshold
for _, row in below_threshold_df.iterrows():
    emplid = row['emplid']
    skill = row['skill']
    
    # Skip if the skill is not in common_skills
    if skill not in common_skills:
        print(f"Skipping skill {skill} - not in common skills")
        continue
    
    # Get employee's skill profile
    try:
        employee_profile = employee_skill_matrix_norm.loc[emplid].values.reshape(1, -1)
    except KeyError:
        print(f"Skipping emplid {emplid} - not found in skill matrix")
        continue
    
    # Calculate cosine similarity between employee and all content
    similarity_scores = cosine_similarity(employee_profile, content_skill_matrix_norm)
    
    # Get content related to this skill
    relevant_content = content_df[content_df['skill'] == skill]
    
    # Get content IDs already consumed by this employee
    consumed_content = consumption_df[consumption_df['emplid'] == emplid]['content_id'].unique()
    
    # Filter out already consumed content
    relevant_content = relevant_content[~relevant_content['content_id'].isin(consumed_content)]
    
    # Skip if no unconsumed content is available for this skill
    if len(relevant_content) == 0:
        print(f"No unconsumed content available for emplid {emplid}, skill {skill}")
        continue
    
    # Get high performers for this skill
    skill_high_performers = high_performers_df[high_performers_df['skill'] == skill]
    
    # Skip if no high performers for this skill
    if len(skill_high_performers) == 0:
        print(f"No high performers found for skill {skill}")
        # Use only cosine similarity for scoring
        content_scores = []
        for _, content in relevant_content.iterrows():
            try:
                # Get cosine similarity score for this content
                content_similarity = similarity_scores[0][content_skill_matrix_norm.index.get_loc(content['content_id'])]
                
                content_scores.append({
                    'content_id': content['content_id'],
                    'score': content_similarity,  # Use only similarity score
                    'employee_skill_score': row['score']
                })
            except KeyError as e:
                print(f"Error processing content_id {content['content_id']}: {str(e)}")
                continue
    else:
        # Calculate content effectiveness score with both completion rate and similarity
        content_scores = []
        for _, content in relevant_content.iterrows():
            try:
                # Calculate completion rate among high performers
                high_performer_completions = consumption_df[
                    (consumption_df['emplid'].isin(skill_high_performers['emplid'])) & 
                    (consumption_df['content_id'] == content['content_id'])
                ]
                completion_rate = len(high_performer_completions) / len(skill_high_performers)
                
                # Get cosine similarity score for this content
                content_similarity = similarity_scores[0][content_skill_matrix_norm.index.get_loc(content['content_id'])]
                
                # Combine completion rate and similarity score
                combined_score = (completion_rate + content_similarity) / 2
                
                content_scores.append({
                    'content_id': content['content_id'],
                    'score': combined_score,
                    'employee_skill_score': row['score']
                })
            except KeyError as e:
                print(f"Error processing content_id {content['content_id']}: {str(e)}")
                continue
    
    # Skip if no valid content scores were calculated
    if not content_scores:
        print(f"No valid content scores for emplid {emplid}, skill {skill}")
        continue
    
    # Get top 5 recommendations (or all if less than 5 available)
    max_recommendations = min(5, len(content_scores))
    top_recommendations = sorted(content_scores, key=lambda x: x['score'], reverse=True)[:max_recommendations]
    
    for rec in top_recommendations:
        recommendations.append({
            'emplid': emplid,
            'skill': skill,
            'content_id': rec['content_id'],
            'recommendation_score': rec['score'],
            'current_skill_score': rec['employee_skill_score']
        })

In [17]:
# Create final recommendations DataFrame
recommendations_df = pd.DataFrame(recommendations)

if len(recommendations_df) > 0:
    # Remove duplicate recommendations (same content for different skills)
    recommendations_df = recommendations_df.drop_duplicates(subset=['emplid', 'content_id'])

    # Sort recommendations by emplid and then by recommendation_score in descending order
    recommendations_df = recommendations_df.sort_values(['emplid', 'recommendation_score'], ascending=[True, False])
    recommendations_df.to_csv('Recommendations/Content/top_5_content_recommendations.csv', index=False)

    # Display first few recommendations
    print("\nFirst few recommendations:")
    print(recommendations_df.head())

    # Display summary statistics
    print("\nSummary statistics:")
    print(f"Total recommendations generated: {len(recommendations_df)}")
    print(f"Number of employees receiving recommendations: {recommendations_df['emplid'].nunique()}")
    print(f"Number of skills covered: {recommendations_df['skill'].nunique()}")

    # Display average recommendation score by skill
    print("\nAverage recommendation score by skill:")
    print(recommendations_df.groupby('skill')['recommendation_score'].mean().sort_values(ascending=False))

    # Display statistics about skills below threshold
    print("\nSkills below threshold (41) statistics:")
    print(f"Total number of skills below threshold: {len(below_threshold_df)}")
    print(f"Number of unique employees with skills below threshold: {below_threshold_df['emplid'].nunique()}")
    print("\nDistribution of skill scores below threshold:")
    print(below_threshold_df['score'].describe())
else:
    print("No recommendations were generated.")


First few recommendations:
       emplid     skill                content_id  recommendation_score  \
16945   23354  Strategy   verified_skills_all.csv              0.800290   
16946   23354  Strategy  667a5eda027461c9d3948aff              0.788194   
16947   23354  Strategy  610416c8dd16b245becc0aef              0.689034   
16948   23354  Strategy  5bdcefc4659e93449792494e              0.661478   
16949   23354  Strategy  5f456ec660e9cc287b1a841b              0.661478   

       current_skill_score  
16945            48.190255  
16946            48.190255  
16947            48.190255  
16948            48.190255  
16949            48.190255  

Summary statistics:
Total recommendations generated: 83693
Number of employees receiving recommendations: 7402
Number of skills covered: 11

Average recommendation score by skill:
skill
Strategy               0.740472
Communication          0.707094
Problem Solving        0.703418
Industry Knowledge     0.668952
Process                0.654225
